In [1]:
import pandas as pd
import praw
from prawcore.exceptions import Forbidden, NotFound, RequestException
import sys

# API Setup

In [2]:
with open(r"pw.txt", 'r') as f:
    pw = f.read()
reddit = praw.Reddit(
    client_id="ee5DffSIb70G-g7EOVuG2A",
    client_secret="NRBMN3dQLyygyRHsknNa69eTWrxXpw",
    password=pw,
    user_agent="USERAGENT",
    username="Progzly",
)

# Dataframe 1

## Recursion Script

In [3]:
df = pd.read_csv(r'userData-filterd.csv')
userArr = df['author'].tolist()

In [4]:
len(userArr)

234935

In [5]:
def get_subreddit_submission_dictionary(username):
    while True:
        try:
            user = reddit.redditor(username)
            posts = user.submissions.new(limit=None)
            subreddit_dict = {'total_count': 0}
            entry_count = 0
            post_array = []
            for post in posts:
                entry_count += 1
                sub = post.subreddit.display_name
                subreddit_dict[sub] = subreddit_dict.get(sub, 0)
                subreddit_dict[sub] = subreddit_dict.get(sub) + 1
                features = {
                    'subreddit': post.subreddit.display_name,
                    'title': post.title,
                    'selftext': post.selftext,
                    'created_utc': post.created_utc,
                    'score': post.score,
                    'upvote_ratio': post.upvote_ratio,
                    'num_comments': post.num_comments
                }
                post_array.append(features)
            subreddit_dict['submission_count'] = entry_count
            return [subreddit_dict, post_array]
        except Forbidden:
            # print(f'Access to {username} denied. Skipping...')
            return [None, None]
        except NotFound:
            # print(f'{username} not found. Skipping...')
            return [None, None]

def get_subreddit_comment_dictionary(username):
    while True:
        try:
            user = reddit.redditor(username)
            comments = user.comments.new(limit=None)
            subreddit_dict = {'total_count': 0}
            entry_count = 0
            subreddit_array = []
            for comment in comments:
                entry_count += 1
                sub = comment.subreddit.display_name
                subreddit_dict[sub] = subreddit_dict.get(sub, 0)
                subreddit_dict[sub] = subreddit_dict.get(sub) + 1
                features = {
                    'subreddit': comment.subreddit.display_name,
                    'body': comment.body,
                    'created_utc': comment.created_utc,
                    'score': comment.score,
                }
                subreddit_array.append(features)
            subreddit_dict['comment_count'] = entry_count
            return [subreddit_dict, subreddit_array]
        except Forbidden:
            # print(f'Access to {username} denied. Skipping...')
            return [None, None]
        except NotFound:
            # print(f'{username} not found. Skipping...')
            return [None, None]

In [6]:
batch_len = 23493
batch1 = userArr[0:batch_len]
batch2 = userArr[batch_len:2*batch_len]
batch3 = userArr[2*batch_len:3*batch_len]
batch4 = userArr[3*batch_len:4*batch_len]
batch5 = userArr[4*batch_len:5*batch_len]
batch6 = userArr[5*batch_len:6*batch_len]
batch7 = userArr[6*batch_len:7*batch_len]
batch8 = userArr[7*batch_len:8*batch_len]
batch9 = userArr[8*batch_len:9*batch_len]
batch10 = userArr[9*batch_len:10*batch_len]
batch11 = userArr[10*batch_len:10*batch_len+5]
batchArr = ['OFFSET',batch1, batch2, batch3, batch4, batch5, batch6, batch7, batch8, batch9, batch10, batch11]

In [7]:
# batchNum = 1
# offset = 2185 - batch_len*(batchNum-1)
file = open('offset.txt', 'r')
offset = int(file.read().strip())
file.close()
file = open('batchnum.txt', 'r')
batchNum = int(file.read().strip())
file.close()
current = batch_len * (batchNum-1) + offset
try:
    for username in batchArr[batchNum][offset:]:
        print(f'Processing #{current}: {username}... ', end='')
        subreddit_posts_all = get_subreddit_submission_dictionary(username)
        subreddit_comments_all = get_subreddit_comment_dictionary(username)
        subreddit_dict_posts = subreddit_posts_all[0]
        subreddit_dict_comments = subreddit_comments_all[0]
        submission_array = subreddit_posts_all[1]
        comment_array = subreddit_comments_all[1]

        if subreddit_dict_posts is None or subreddit_dict_comments is None or submission_array is None or comment_array is None:
            print(f'Denied :(')
            current += 1
            continue

        posts_count_df = pd.DataFrame(list(subreddit_dict_posts.items()), columns=['Subreddit', 'Post Count'])
        comments_count_df = pd.DataFrame(list(subreddit_dict_comments.items()), columns=['Subreddit', 'Comment Count'])
        posts_meta_df = pd.DataFrame(submission_array, columns=['subreddit', 'title', 'selftext', 'created_utc', 'score', 'upvote_ratio', 'num_comments'])
        comments_meta_df = pd.DataFrame(comment_array, columns=['subreddit', 'body', 'created_utc', 'score'])

        posts_count_df.to_csv(f'count_data/batch{batchNum}/{username}_sub_posts_count.csv', index=False)
        comments_count_df.to_csv(f'count_data/batch{batchNum}/{username}_sub_comments_count.csv', index=False)
        posts_meta_df.to_csv(f'meta_data/batch{batchNum}/{username}_metadata_posts.csv', index=False)
        comments_meta_df.to_csv(f'meta_data/batch{batchNum}/{username}_metadata_comments.csv', index=False)

        print(f'Done!')
        current += 1
    print(f'Batch {batchNum} completed.')
    offset = current - batch_len * (batchNum-1+1)
    file = open(f'offset.txt', 'w')
    file.write(str(offset))
    file.close()
    print(f'Offset saved: {offset}')
    batchNum += 1
    file = open(f'batchnum.txt', 'w')
    file.write(str(batchNum))
    file.close()
    # sys.exit(0)
except KeyboardInterrupt:
    print("\nManual interruption detected. Saving offset...")
    offset = current - batch_len * (batchNum-1)
    file = open(f'offset.txt', 'w')
    file.write(str(offset))
    file.close()
    print(f'\nOffset saved: {offset}')
    sys.exit(0)
except RequestException:
    print("\nRequest exception detected. Saving offset...")
    offset = current - batch_len * (batchNum-1)
    file = open(f'offset.txt', 'w')
    file.write(str(offset))
    file.close()
    print(f'\nOffset saved: {offset}')
    sys.exit(1)

Processing #145058: ARquantam... Done!
Processing #145059: MentalOverhaul... Done!
Processing #145060: Groaawr... Done!
Processing #145061: pending_zone... Done!
Processing #145062: lostallhope4... Done!
Processing #145063: Ro_guee... Done!
Processing #145064: feddser123... Denied :(
Processing #145065: throwawayy84728474... Done!
Processing #145066: Kraabsito... Done!
Processing #145067: WorkingOnItEveryDay... Done!
Processing #145068: throwaway181000... Done!
Processing #145069: adirbegerano... Done!
Processing #145070: xyveris... Done!
Processing #145071: london_beckoned... Done!
Processing #145072: shesaidnoooope... Done!
Processing #145073: Dovitk... Done!
Processing #145074: TheGuy1214... Done!
Processing #145075: -Ette-... Done!
Processing #145076: hippie-suave... Done!
Processing #145077: Leonsandcastle18... Done!
Processing #145078: veeto76... Done!
Processing #145079: Orion66... Done!
Processing #145080: Kitsunefae... Done!
Processing #145081: brainofcain... Done!
Processing 

SystemExit: 0

c:\Users\NickB\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


**Common Name Themes:** "takemetoabetterplace", "throwmeaway", "suicide", "to", "suicidethrowaway", "UglyTA", "burner", "throaway", "ifuckeduprealnice", "suicidal", "sad_and_lonely", "pleasehelpme", "Kiamdepressed", "thefinal_throwaway", "too_crushed_to_live", "too", "trans", "please_helmyfriend", "forget", "life". "hurts", "everday", "tired", "why", "throaway", "ohgodhelpmepl", "iamnotdyingtoday", "FailingToFindAReason", "Cursedwithsuck", "cantwait710", "canthandlenotworking", "nowherelefttoturn", "lookingforanewway", "givenuponbeinghappy", "feelinglike_shit", "suicidewatchthrowaw", "lesbiansweetheart95", "donotuseagain", "thishurt", "Rememberdp", "hadenough123e", "pointlessasfuck", "Absolutetrainwreck", "Feelinginadequate1", "ihatethisshit12", "suicide_throw_away66", "", "", "", "", "", "", "", ""
<br>
**Correlation Example:** ifuckeduprealnice (*r/SuicideWatch & r/AskGayBros*)

In [8]:
batch6.index('Iridektm300')

3997

In [8]:
file = open('batchnum.txt', 'r')
batchNum = int(file.read().strip())
file.close()

In [9]:
# Run in case of error or Keyboard interrupt
## 2463
current = 124687
offset = current - batch_len * (batchNum-1)
file = open(f'offset.txt', 'w')
file.write(str(offset))
file.close()
offset

7222

In [ ]:
file = open(f'batchNum.txt', 'w')
file.write(str(batchNum))
file.close()
batchNum

2

## Singlar User Tests

In [9]:
df = pd.read_csv(r'userData-filterd.csv')
userArr = df['author'].tolist()

In [10]:
username = userArr[12]

In [11]:
user = reddit.redditor(username)

In [12]:
# Get Dictionary of Subreddit Count and Posts
posts = user.submissions.new(limit=None)
subreddit_dict = {'username': username}
post_count = 0
for post in posts:
    post_count += 1
    sub = post.subreddit.display_name
    subreddit_dict[sub] = subreddit_dict.get(sub, 0)
    # print(type(subreddit_dict[sub]))
    subreddit_dict[sub] = subreddit_dict.get(sub) + 1
print(post_count)
subreddit_dict

6


{'username': 'GuigzForAll',
 'atheism': 1,
 'programming': 1,
 'AskReddit': 2,
 'giveaways': 1,
 'SuicideWatch': 1}

In [13]:
comments = user.comments.new(limit=None)
subreddit_dict = {}
comment_count = 0
print(comment_count)
for comment in comments:
    comment_count += 1
    sub = comment.subreddit.display_name
    subreddit_dict[sub] = subreddit_dict.get(sub, 0)
    # print(type(subreddit_dict[sub]))
    subreddit_dict[sub] = subreddit_dict.get(sub) + 1
print(comment_count)
subreddit_dict

0
1044


{'business': 42,
 'WTF': 74,
 'canada': 38,
 'AskReddit': 129,
 'atheism': 224,
 'politics': 63,
 'offbeat': 40,
 'technology': 55,
 'SuicideWatch': 13,
 'reddit.com': 70,
 'programming': 65,
 'pics': 65,
 'nsfw': 13,
 'funny': 70,
 'science': 57,
 'wow': 1,
 'gonewild': 6,
 'philosophy': 3,
 'giveaways': 2,
 'lovereddit': 3,
 'self': 1,
 'law': 1,
 'Boobies': 1,
 'worldnews': 8}

In [18]:
test_dict = {'username':'TEST_USERNAME','test_col1': 1, 'test_col2': 2}
test_df = pd.DataFrame([subreddit_dict, test_dict])
test_df

,username,atheism,programming,AskReddit,giveaways,SuicideWatch,test_col1,test_col2
0,GuigzForAll,1.0,1.0,2.0,1.0,1.0,NaN,NaN
1,TEST_USERNAME,NaN,NaN,NaN,NaN,NaN,1.0,2.0


In [14]:
test_df = None

In [51]:
posts = user.submissions.new(limit=None)
for post in posts:
    features = {
        'author': post.author.name,
        'subreddit': post.subreddit.display_name,
        'title': post.title,
        'selftext': post.selftext,
        'created_utc': post.created_utc,
        'score': post.score,
        'upvote_ratio': post.upvote_ratio,
        'num_comments': post.num_comments
    }
    print(features)

{'author': 'GuigzForAll', 'subreddit': 'atheism', 'title': 'James Randi tests an aura reader.  Guess who wins?', 'selftext': '', 'created_utc': 1232236426.0, 'score': 9, 'upvote_ratio': 0.64, 'num_comments': 12}
{'author': 'GuigzForAll', 'subreddit': 'programming', 'title': 'Ask Proggit: Ever wondered "Why the hell is this working?"?', 'selftext': '', 'created_utc': 1231779274.0, 'score': 7, 'upvote_ratio': 0.62, 'num_comments': 17}
{'author': 'GuigzForAll', 'subreddit': 'AskReddit', 'title': 'Ask Reddit: Is english just integrated in japanese or what?', 'selftext': '', 'created_utc': 1231027927.0, 'score': 1, 'upvote_ratio': 0.6, 'num_comments': 6}
{'author': 'GuigzForAll', 'subreddit': 'giveaways', 'title': "Iceland's economy", 'selftext': '', 'created_utc': 1230444117.0, 'score': 9, 'upvote_ratio': 0.74, 'num_comments': 7}
{'author': 'GuigzForAll', 'subreddit': 'SuicideWatch', 'title': 'Thank you, whoever thought of this.', 'selftext': '', 'created_utc': 1229485535.0, 'score': 29, '

In [19]:
# Get Dictionary of Subreddit Count and Posts
posts = user.submissions.new(limit=None)
subreddit_dict = {'username': username}
post_count = 0
post_array = []
for post in posts:
    post_count += 1
    sub = post.subreddit.display_name
    subreddit_dict[sub] = subreddit_dict.get(sub, 0)
    # print(type(subreddit_dict[sub]))
    subreddit_dict[sub] = subreddit_dict.get(sub) + 1
    features = {
        'author': post.author.name,
        'subreddit': post.subreddit.display_name,
        'title': post.title,
        'selftext': post.selftext,
        'created_utc': post.created_utc,
        'score': post.score,
        'upvote_ratio': post.upvote_ratio,
        'num_comments': post.num_comments
    }
    post_array.append(features)
count_feature_arr = [subreddit_dict, post_array]
count_feature_arr[1]

[{'author': 'GuigzForAll',
  'subreddit': 'atheism',
  'title': 'James Randi tests an aura reader.  Guess who wins?',
  'selftext': '',
  'created_utc': 1232236426.0,
  'score': 9,
  'upvote_ratio': 0.64,
  'num_comments': 12},
 {'author': 'GuigzForAll',
  'subreddit': 'programming',
  'title': 'Ask Proggit: Ever wondered "Why the hell is this working?"?',
  'selftext': '',
  'created_utc': 1231779274.0,
  'score': 5,
  'upvote_ratio': 0.59,
  'num_comments': 17},
 {'author': 'GuigzForAll',
  'subreddit': 'AskReddit',
  'title': 'Ask Reddit: Is english just integrated in japanese or what?',
  'selftext': '',
  'created_utc': 1231027927.0,
  'score': 1,
  'upvote_ratio': 0.6,
  'num_comments': 6},
 {'author': 'GuigzForAll',
  'subreddit': 'giveaways',
  'title': "Iceland's economy",
  'selftext': '',
  'created_utc': 1230444117.0,
  'score': 9,
  'upvote_ratio': 0.72,
  'num_comments': 7},
 {'author': 'GuigzForAll',
  'subreddit': 'SuicideWatch',
  'title': 'Thank you, whoever thought o